# Frameworks comparativa · LangChain · LlamaIndex · SmolAgents

> **LAB del módulo de Frameworks**: el mismo caso resuelto con los tres frameworks, para tener una intuición real de cuándo usar cada uno.

## Caso de uso

Tenemos un corpus pequeño (5 documentos cortos sobre IA en 2026). El usuario pregunta algo, y cada framework debe responder usando ese corpus como contexto.

No es un RAG completo (eso lo veremos en el módulo de RAG). Es el **"hello world"** de los 3 frameworks: cómo se ve el código, qué piezas tiene cada uno, dónde brilla cada uno.

## Lo que vamos a comparar

| Eje | Qué medimos |
|---|---|
| Líneas de código | Cuán verboso es el setup |
| Modelo mental | Chain · Index · Agent |
| Curva de entrada | Qué necesitas saber para empezar |
| Sweet spot | Para qué tipo de proyecto encaja mejor |

## Stack 2026

- `langchain>=1.0` (LCEL · Runnables). En LangChain 1.x los agentes se construyen con `create_agent` sobre LangGraph y las chains antiguas viven en `langchain-classic`; aquí solo usamos LCEL, que sigue en `langchain-core`
- `llama-index>=0.14`
- `smolagents>=1.20`
- Validado el 2026-10-14 con langchain 1.4.5, langchain-openai 1.7.2, llama-index 0.14.25, smolagents 1.26.0, faiss-cpu 1.15.1
- LLM: `gpt-5-mini` (cualquier endpoint OpenAI compatible vale)
- Embeddings: `text-embedding-3-small`


## Setup

In [1]:
%pip install -q \
  "langchain>=1.0" "langchain-core>=1.0" "langchain-openai>=1.0" "langchain-community>=0.4" \
  "llama-index>=0.14" "llama-index-llms-openai" "llama-index-embeddings-openai" \
  "smolagents>=1.20" \
  "faiss-cpu" "openai"

In [2]:
import os, getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key: ")

LLM_MODEL = "gpt-5-mini"
EMBED_MODEL = "text-embedding-3-small"

## Corpus compartido

5 docs cortos en memoria. Es el mismo input para los 3 frameworks: así la comparación es honesta.

In [3]:
DOCS = [
    "GPT-5 es un modelo de OpenAI lanzado en 2025. Su variante mini (gpt-5-mini) tiene un coste muy bajo por token y es la opción habitual para producción cuando no necesitas el modelo full.",
    "Claude 4 Sonnet es el modelo medio de Anthropic en 2026. Destaca en razonamiento de código y ventana de contexto larga (1M tokens en la variante 1M).",
    "Gemini 2.5 Pro es el modelo flagship de Google en 2026. Tiene capacidades multimodales nativas (texto, imagen, audio, vídeo) y ventana de 2M tokens.",
    "DeepSeek V3 es un modelo open-source competitivo con los cerrados en muchos benchmarks. Su licencia permite uso comercial y se puede self-hostear.",
    "Llama 4 es la familia open de Meta lanzada en 2026. Incluye variantes densas y MoE (Mixture of Experts) y es la base de muchos despliegues on-prem.",
]

QUERY = "¿Qué modelo recomiendas para producción si quiero bajo coste?"

## Versión 1 · LangChain

**Modelo mental**: chain explícita. Tú compones piezas con `|` (LCEL).

Pasos:
1. Embeddings + vectorstore (FAISS)
2. Retriever
3. Prompt + LLM
4. Chain con LCEL

In [4]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

embeddings = OpenAIEmbeddings(model=EMBED_MODEL)
store = FAISS.from_texts(DOCS, embeddings)
retriever = store.as_retriever(search_kwargs={"k": 3})

prompt = ChatPromptTemplate.from_template(
    "Responde a la pregunta usando solo el contexto.\n\nContexto:\n{context}\n\nPregunta: {question}"
)
llm = ChatOpenAI(model=LLM_MODEL, temperature=0)

chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

answer_lc = chain.invoke(QUERY)
print("LangChain ·", answer_lc)

LangChain · Recomiendo gpt-5-mini, porque tiene un coste muy bajo por token y es la opción habitual para producción cuando no necesitas el modelo full.


## Versión 2 · LlamaIndex

**Modelo mental**: index. Cargas docs, los indexas, le pides una `query_engine` y le hablas.

Pasos:
1. Settings globales (LLM, embeddings)
2. Documents → VectorStoreIndex
3. as_query_engine() y listo

In [5]:
from llama_index.core import VectorStoreIndex, Document, Settings
from llama_index.llms.openai import OpenAI as LIOpenAI
from llama_index.embeddings.openai import OpenAIEmbedding as LIEmbedding

Settings.llm = LIOpenAI(model=LLM_MODEL, temperature=0)
Settings.embed_model = LIEmbedding(model=EMBED_MODEL)

documents = [Document(text=d) for d in DOCS]
index = VectorStoreIndex.from_documents(documents)
query_engine = index.as_query_engine(similarity_top_k=3)

answer_li = query_engine.query(QUERY)
print("LlamaIndex ·", answer_li)

LlamaIndex · Recomiendo gpt-5-mini. Tiene un coste por token muy bajo y suele ser la opción habitual para producción cuando no se necesita el modelo full.

Si prefieres una alternativa open-source y con control total (self-host), considera DeepSeek V3, que permite uso comercial y autoalojamiento. Gemini 2.5 Pro es más orientado a capacidades multimodales y ventana grande, pero no es la opción obvia si tu prioridad principal es minimizar costes.


## Versión 3 · SmolAgents

**Modelo mental**: agent-first. SmolAgents asume que quieres un **agente** que ejecuta código y llama a tools.

Como su sweet spot no es el retrieval puro, le damos un `tool` que busca en nuestro corpus, y es el agente quien decide si lo usa.

Ojo, porque esa decisión es del modelo. Medido el 2026-10-14 con `gpt-5-mini`: pasándole solo `QUERY`, el agente **no usó la tool en 0 de 5 ejecuciones** y respondió de memoria (recomendando `gpt-3.5-turbo` o Llama 2, que no están en el corpus). Diciéndole en la tarea que responda con el corpus interno, la usó en 3 de 3 y recomendó `gpt-5-mini`. En LangChain y LlamaIndex el retrieval es un paso fijo del pipeline; en un agente es una decisión más.

Pasos:
1. Tool de búsqueda sobre el corpus (similitud coseno con embeddings de OpenAI)
2. CodeAgent con la tool
3. Run, indicando en la tarea que responda con el corpus

In [6]:
import numpy as np
from openai import OpenAI as OAI
from smolagents import CodeAgent, OpenAIServerModel, tool

_oai = OAI()

def _embed(texts):
    out = _oai.embeddings.create(model=EMBED_MODEL, input=texts)
    return np.array([d.embedding for d in out.data])

_doc_vecs = _embed(DOCS)

@tool
def search_corpus(query: str, k: int = 3) -> str:
    """Busca en el corpus interno los k documentos más relevantes para la query.

    Args:
        query: Pregunta del usuario.
        k: Número de documentos a devolver.
    """
    qv = _embed([query])[0]
    sims = _doc_vecs @ qv / (np.linalg.norm(_doc_vecs, axis=1) * np.linalg.norm(qv))
    top = np.argsort(-sims)[:k]
    return "\n\n".join(DOCS[i] for i in top)

model = OpenAIServerModel(model_id=LLM_MODEL)
agent = CodeAgent(tools=[search_corpus], model=model)

answer_sa = agent.run(f"Responde usando nuestro corpus interno (tool `search_corpus`). Pregunta: {QUERY}")
print("SmolAgents ·", answer_sa)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Responde usando nuestro corpus interno (tool `search_corpus`). Pregunta: ¿Qué modelo recomiendas para           │
│ producción si quiero bajo coste?                                                                                │
│                                                                                                                 │
╰─ OpenAIModel - gpt-5-mini ──────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  result = search_corpus(query="¿Qué modelo recomiendas para producción si quiero bajo coste?", k=5)               
  print(result)                                                                                                    
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
GPT-5 es un modelo de OpenAI lanzado en 2025. Su variante mini (gpt-5-mini) tiene un coste muy bajo por token y es 
la opción habitual para producción cuando no necesitas el modelo full.

DeepSeek V3 es un modelo open-source competitivo con los cerrados en muchos benchmarks. Su licencia permite uso 
comercial y se puede self-hostear.

Gemini 2.5 Pro es el modelo flagship de Google en 2026. Tiene capacidades multimodales nativas (texto, imagen, 
audio, vídeo) y ventana de 2M tokens.

Llama 4 es la familia open de Meta lanzada en 2026. Incluye variantes densas y MoE (Mixture of Experts) y es la 
base de muchos despliegues on-prem.

Claude 4 Sonnet es el modelo medio de Anthropic en 2026. Destaca en razonamiento de código y ventana de contexto 
larga (1M tokens en la variante 1M).

Out: None

[Step 1: Duration 9.47 seconds| Input tokens: 2,050 | Output tokens: 1,251]

...

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  respuesta = (                                                                                                    
      "Resumen y recomendación para bajo coste en producción:\n\n"                                                 
      "1) Opción gestionada con menor coste por token:\n"                                                          
      "- gpt-5-mini: Recomendado si buscas el menor coste por token en un servicio gestionado. Es la opción        
  habitual para producción cuando no necesitas la versión full.\n\n"                                               
      "2) Opciones self-host (mínimo coste operativo a largo plazo si puedes asumir infra e ingeniería):\n"        
      "- DeepSeek V3: Open-source, licencia que permite uso comercial y se puede self-hostear. Buena relación      
  coste/beneficio.\n"                                                                                              
      "- Llama 4 (variante densa): Familia open de Meta; versión densa suele ser más sencilla y barata de          
  desplegar on-prem que variantes MoE.\n\n"                                                                        
      "3) Cuando NO elegir low-cost y por qué:\n"                                                                  
      "- Gemini 2.5 Pro y Claude 4 Sonnet ofrecen capacidades superiores (multimodalidad, ventanas de contexto     
  enormes) pero son más caros. Úsalos solo si necesitas esas capacidades.\n\n"                                     
      "Recomendación práctica según tu caso:\n"                                                                    
      "- Si prioridad absoluta es bajo coste y quieres servicio gestionado → gpt-5-mini.\n"                        
      "- Si quieres evitar costes recurrentes de proveedor y puedes invertir en infra/operaciones → DeepSeek V3 o  
  Llama 4 (self-host, cuantizar y optimizar despliegue para ahorrar costes).\n\n"                                  
      "Pasos siguientes sugeridos:\n"                                                                              
      "1. Probar un PoC con gpt-5-mini (hosted) para medir coste/latencia y calidad.\n"                            
      "2. Si los costes recurrentes son altos, correr un benchmark con DeepSeek V3 y Llama 4 en tu infra (usando   
  cuantización) y comparar coste total de propiedad.\n"                                                            
      "3. Elegir la opción que cumpla tus requisitos de latencia, privacidad y coste.\n\n"                         
      "Si quieres, puedo ayudarte a diseñar el PoC (pruebas, métricas a medir y configuración de despliegue)       
  según tu tráfico esperado y restricciones."                                                                      
  )                                                                                                                
  print(respuesta)                                                                                                 
  final_answer(respuesta)                                                                                          
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Resumen y recomendación para bajo coste en producción:

1) Opción gestionada con menor coste por token:
- gpt-5-mini: Recomendado si buscas el menor coste por token en un servicio gestionado. Es la opción habitual para 
producción cuando no necesitas la versión full.

2) Opciones self-host (mínimo coste operativo a largo plazo si puedes asumir infra e ingeniería):
- DeepSeek V3: Open-source, licencia que permite uso comercial y se puede self-hostear. Buena relación 
coste/beneficio.
- Llama 4 (variante densa): Familia open de Meta; versión densa suele ser más sencilla y barata de desplegar 
on-prem que variantes MoE.

3) Cuando NO elegir low-cost y por qué:
- Gemini 2.5 Pro y Claude 4 Sonnet ofrecen capacidades superiores (multimodalidad, ventanas de contexto enormes) 
pero son más caros. Úsalos solo si necesitas esas capacidades.

Recomendación práctica según tu caso:
- Si prioridad absoluta es bajo coste y quieres servicio gestionado → gpt-5-mini.
- Si quieres evitar costes recurrentes de proveedor y puedes invertir en infra/operaciones → DeepSeek V3 o Llama 4 
(self-host, cuantizar y optimizar despliegue para ahorrar costes).

Pasos siguientes sugeridos:
1. Probar un PoC con gpt-5-mini (hosted) para medir coste/latencia y calidad.
2. Si los costes recurrentes son altos, correr un benchmark con DeepSeek V3 y Llama 4 en tu infra (usando 
cuantización) y comparar coste total de propiedad.
3. Elegir la opción que cumpla tus requisitos de latencia, privacidad y coste.

Si quieres, puedo ayudarte a diseñar el PoC (pruebas, métricas a medir y configuración de despliegue) según tu 
tráfico esperado y restricciones.

Final answer: Resumen y recomendación para bajo coste en producción:

1) Opción gestionada con menor coste por token:
- gpt-5-mini: Recomendado si buscas el menor coste por token en un servicio gestionado. Es la opción habitual para 
producción cuando no necesitas la versión full.

2) Opciones self-host (mínimo coste operativo a largo plazo si puedes asumir infra e ingeniería):
- DeepSeek V3: Open-source, licencia que permite uso comercial y se puede self-hostear. Buena relación 
coste/beneficio.
- Llama 4 (variante densa): Familia open de Meta; versión densa suele ser más sencilla y barata de desplegar 
on-prem que variantes MoE.

3) Cuando NO elegir low-cost y por qué:
- Gemini 2.5 Pro y Claude 4 Sonnet ofrecen capacidades superiores (multimodalidad, ventanas de contexto enormes) 
pero son más caros. Úsalos solo si necesitas esas capacidades.

Recomendación práctica según tu caso:
- Si prioridad absoluta es bajo coste y quieres servicio gestionado → gpt-5-mini.
- Si quieres evitar costes recurrentes de proveedor y puedes invertir en infra/operaciones → DeepSeek V3 o Llama 4 
(self-host, cuantizar y optimizar despliegue para ahorrar costes).

Pasos siguientes sugeridos:
1. Probar un PoC con gpt-5-mini (hosted) para medir coste/latencia y calidad.
2. Si los costes recurrentes son altos, correr un benchmark con DeepSeek V3 y Llama 4 en tu infra (usando 
cuantización) y comparar coste total de propiedad.
3. Elegir la opción que cumpla tus requisitos de latencia, privacidad y coste.

Si quieres, puedo ayudarte a diseñar el PoC (pruebas, métricas a medir y configuración de despliegue) según tu 
tráfico esperado y restricciones.

[Step 2: Duration 10.08 seconds| Input tokens: 4,471 | Output tokens: 2,547]

SmolAgents · Resumen y recomendación para bajo coste en producción:

1) Opción gestionada con menor coste por token:
- gpt-5-mini: Recomendado si buscas el menor coste por token en un servicio gestionado. Es la opción habitual para producción cuando no necesitas la versión full.

2) Opciones self-host (mínimo coste operativo a largo plazo si puedes asumir infra e ingeniería):
- DeepSeek V3: Open-source, licencia que permite uso comercial y se puede self-hostear. Buena relación coste/beneficio.
- Llama 4 (variante densa): Familia open de Meta; versión densa suele ser más sencilla y barata de desplegar on-prem que variantes MoE.

3) Cuando NO elegir low-cost y por qué:
- Gemini 2.5 Pro y Claude 4 Sonnet ofrecen capacidades superiores (multimodalidad, ventanas de contexto enormes) pero son más caros. Úsalos solo si necesitas esas capacidades.

Recomendación práctica según tu caso:
- Si prioridad absoluta es bajo coste y quieres servicio gestionado → gpt-5-mini.
- Si quieres evitar costes 

### Contraejemplo · un agente sin tools

Mismo `CodeAgent`, pero **sin ninguna tool** y con una pregunta que necesita datos en vivo (precios de Bitcoin).
El agente escribe código Python, pero se ejecuta en un intérprete local restringido: solo puede importar
módulos de una lista blanca (`math`, `datetime`, `re`...), sin `requests` ni acceso a internet.

Fíjate en qué hace: intenta `import requests`, el intérprete se lo bloquea y acaba reconociendo que no puede (o, con preguntas ambiguas, pide aclaraciones; con modelos peores, se inventa precios).
La lección: **un agente vale lo que valen sus tools**. Si quieres datos reales, dale una tool que los traiga.


In [7]:
agent = CodeAgent(tools=[], model=model, max_steps=3)   # acotamos pasos: sin tools ni red, el dato no existe
answer_sa = agent.run("Dame el precio de cierre en USD de Bitcoin de cada uno de los últimos 3 días.")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Dame el precio de cierre en USD de Bitcoin de cada uno de los últimos 3 días.                                   │
│                                                                                                                 │
╰─ OpenAIModel - gpt-5-mini ──────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  # Script: Obtener precio de cierre diario (USD) de Bitcoin - últimos 3 días completos (UTC)                      
  # Requisitos: pip install requests                                                                               
  # Uso: ejecutar en tu máquina: python btc_last3_closes.py                                                        
                                                                                                                   
  import datetime                                                                                                  
  import collections                                                                                               
  import json                                                                                                      
  import textwrap                                                                                                  
                                                                                                                   
  # Este bloque es solo informativo y no hace llamadas HTTP aquí porque                                            
  # en este entorno no hay acceso a Internet. Copia abajo el bloque marcado                                        
  # como `fetch_code` en un archivo .py y ejecútalo en tu equipo con Internet.                                     
                                                                                                                   
  fetch_code = r'''                                                                                                
  import requests                                                                                                  
  from datetime import datetime, timezone                                                                          
  from collections import defaultdict                                                                              
                                                                                                                   
  # Pedimos 4 días para asegurarnos de cubrir los últimos 3 días completos                                         
  resp = requests.get(                                                                                             
      "https://api.coingecko.com/api/v3/coins/bitcoin/market_chart",                                               
      params={"vs_currency": "usd", "days": "4", "interval": "hourly"},                                            
      timeout=10                                                                                                   
  )                                                                                                                
  resp.raise_for_status()                                                                                          
  data = resp.json()                                                                                               
                                                                                                                   
  # 'prices' es una lista de [timestamp_ms, price]                                                                 
  prices = data.get("prices", [])                                                                                  
                                                                                                                   
  # Agrupar por fecha UTC (YYYY-MM-DD) y tomar el último precio de cada día                                        
  by_date = defaultdict(list)                                                                                      
  for ts_ms, price in prices:                                                                                      
      dt = datetime.fromtimestamp(ts_ms/1000.0, tz=timezo

Warning to user: Code execution failed due to an unauthorized import - Consider passing said import under 
`additional_authorized_imports` when initializing your CodeAgent.

Code execution failed at line 'import json' due to: InterpreterError: Import of json is not allowed. Authorized 
imports are: ['math', 'time', 'itertools', 'datetime', 'collections', 'statistics', 'stat', 're', 'queue', 
'unicodedata', 'random']

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  # Script que imprime instrucciones y el código a ejecutar localmente.                                            
  # No importa módulos no autorizados en este entorno.                                                             
  # Copia el bloque marcado como `fetch_code` y ejecútalo en tu equipo con Internet.                               
                                                                                                                   
  fetch_code = '''#!/usr/bin/env python3                                                                           
  # Obtener precio de cierre diario (USD) de Bitcoin - últimos 3 días completos (UTC)                              
  # Requisitos: pip install requests                                                                               
  # Uso: python btc_last3_closes.py                                                                                
                                                                                                                   
  import requests                                                                                                  
  from datetime import datetime, timezone                                                                          
  from collections import defaultdict                                                                              
                                                                                                                   
  def main():                                                                                                      
      # Pedimos 4 días para asegurarnos de cubrir los últimos 3 días completos                                     
      resp = requests.get(                                                                                         
          "https://api.coingecko.com/api/v3/coins/bitcoin/market_chart",                                           
          params={"vs_currency": "usd", "days": "4", "interval": "hourly"},                                        
          timeout=10                                                                                               
      )                                                                                                            
      resp.raise_for_status()                                                                                      
      data = resp.json()                                                                                           
                                                                                                                   
      # 'prices' es una lista de [timestamp_ms, price]                                                             
      prices = data.get("prices", [])                                                                              
                                                                                                                   
      # Agrupar por fecha UTC (YYYY-MM-DD) y tomar el último precio de cada día                                    
      by_date = defaultdict(list)                                                                                  
      for ts_ms, price in prices:                                                                                  
          dt = datetime.fromtimestamp(ts_ms / 1000.0, tz=timezone.utc)                                             
          day = dt.date().isoformat()                                                                              
          by_date[day].append((dt, price))                                                                         
                                                                                                                   
      # Para cada día, ordenamos por datetime y tomamos e

Execution logs:
No puedo acceder a Internet desde este entorno. Debajo tienes un script listo para ejecutar en tu máquina. Cópialo 
en un archivo, por ejemplo 'btc_last3_closes.py', instala requests si es necesario (pip install requests) y 
ejecuta:

python btc_last3_closes.py

El script devuelve el precio de cierre (último precio disponible) en USD de los últimos 3 días completos según UTC 
(excluye el día en curso UTC). Si quieres que incluya hoy o que use otra fuente, dime y lo adapto.
----- Código para copiar y ejecutar -----
#!/usr/bin/env python3
# Obtener precio de cierre diario (USD) de Bitcoin - últimos 3 días completos (UTC)
# Requisitos: pip install requests
# Uso: python btc_last3_closes.py

import requests
from datetime import datetime, timezone
from collections import defaultdict

def main():
    # Pedimos 4 días para asegurarnos de cubrir los últimos 3 días completos
    resp = requests.get(
        "https://api.coingecko.com/api/v3/coins/bitcoin/market_chart",
        params={"vs_currency": "usd", "days": "4", "interval": "hourly"},
        timeout=10
    )
    resp.raise_for_status()
    data = resp.json()

    # 'prices' es una lista de [timestamp_ms, price]
    prices = data.get("prices", [])

    # Agrupar por fecha UTC (YYYY-MM-DD) y tomar el último precio de cada día
    by_date = defaultdict(list)
    for ts_ms, price in prices:
        dt = datetime.fromtimestamp(ts_ms / 1000.0, tz=timezone.utc)
        day = dt.date().isoformat()
        by_date[day].append((dt, price))

    # Para cada día, ordenamos por datetime y tomamos el último precio (cierre)
    closes = {}
    for day, entries in by_date.items():
        entries.sort(key=lambda x: x[0])
        closes[day] = entries[-1][1]

    # Ordenar fechas y seleccionar los últimos 3 días completos (excluyendo la fecha UTC actual)
    today_utc = datetime.now(timezone.utc).date().isoformat()
    dates_sorted = sorted(closes.keys())
    dates_filtered = [d for d in dates_sorted if d != today_utc]
    last3 = dates_filtered[-3:]

    if not last3:
        print("No se encontraron suficientes datos. Intenta aumentar 'days' en la consulta.")
        return

    # Imprimir resultados en orden más reciente -> más antiguo
    for d in reversed(last3):
        print(f"{d}: {closes[d]:,.2f} USD")

if __name__ == "__main__":
    main()

----- Fin del código -----

Final answer: No puedo acceder a Internet desde este entorno. Debajo tienes un script listo para ejecutar en tu 
máquina. Cópialo en un archivo, por ejemplo 'btc_last3_closes.py', instala requests si es necesario (pip install 
requests) y ejecuta:

python btc_last3_closes.py

El script devuelve el precio de cierre (último precio disponible) en USD de los últimos 3 días completos según UTC 
(excluye el día en curso UTC). Si quieres que incluya hoy o que use otra fuente, dime y lo adapto.

Código incluido arriba.

[Step 2: Duration 13.01 seconds| Input tokens: 5,814 | Output tokens: 4,353]

## Comparativa

Si has llegado hasta aquí ejecutando, los tres acaban recomendando `gpt-5-mini` (es lo que dice el corpus), pero SmolAgents solo cuando le pedimos que use su tool: sin esa indicación responde de memoria. El **camino** ha sido distinto.

| Eje | LangChain | LlamaIndex | SmolAgents |
|---|---|---|---|
| Líneas de código del setup | ~12 | ~7 | ~15 (más glue) |
| Modelo mental | Chain (LCEL `\|`) | Index + QueryEngine | Agent + Tools |
| Verbosidad | Media (explícito) | Baja (todo "just works") | Media (pides tool) |
| Sweet spot | Pipelines complejos · routing | RAG · datos propios | Agentes que ejecutan código |
| Curva | Plana si entiendes LCEL | La más rápida para empezar | Hay que pensar en términos de agente |
| Comunidad | Más grande · más integraciones | Sólida en RAG | Pequeña · creciendo |

## Cuándo usar cuál

- **Tu app es básicamente RAG sobre documentos propios** : LlamaIndex. Te ahorra código.
- **Tu app tiene varios pasos · routing · múltiples LLMs · memoria conversacional** : LangChain. Es lo que mejor hace.
- **Tu app es un agente que ejecuta código · tools complejas · razonamiento iterativo** : SmolAgents (o LangGraph, que veremos en el módulo de Agentes con LangChain de la clase 5).

## Y la realidad

En proyectos de verdad **se mezclan**. Es muy común usar LlamaIndex como retriever dentro de una chain de LangChain, o un agente de SmolAgents que llama a un query_engine de LlamaIndex como tool. Elige por **dolor**, no por marca.
